# QM640 · 04b Jira duplicate (D2) and cause (D6) test sets and I1 search
Completes notebook 04, using about 5 GB of memory, so a standard CPU runtime is enough. About 5–10 minutes. *Runtime → Run all*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')

## 1. Settings

In [ ]:
import json, time, gc, datetime, numpy as np, pandas as pd, pyarrow.parquet as pq
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.preprocessing import normalize
from statsmodels.stats.proportion import proportion_confint
JIRA = DATA_ROOT/'processed'/'jira'; RESULTS = DATA_ROOT/'results'; OUTP = DATA_ROOT/'processed'/'jira_tests'
SEED, N_TEST, TOP_K = 640, 385, 10
rng = np.random.default_rng(SEED)
def wilson(k, n): lo, hi = proportion_confint(k, n, method='wilson'); return f'{100*lo:.1f}–{100*hi:.1f}'
FILES = sorted(JIRA.glob('*.parquet'))

## 2. Same time split as notebook 04
The test period is the latest 20% of issues, identical to notebook 04.

In [ ]:
created = pd.concat([pd.to_datetime(pd.read_parquet(f, columns=['created'])['created'], errors='coerce', utc=True) for f in FILES])
created = created[created >= pd.Timestamp('2000-01-01', tz='UTC')]
q80 = created.quantile(.8); del created; gc.collect()
print('test period starts', q80)

## 3. Test sets for D2 (duplicates) and D6 (causes)
A test issue is used only if the issue it duplicates, or the issue that caused it, was created earlier. 385 are sampled for each decision (seed 640).

In [ ]:
KINDS = {'D2_duplicate': lambda t, l: 'duplicate' in t,
         'D6_cause': lambda t, l: 'caused by' in l or (('cause' in t or 'incident' in t) and l.startswith('is caused'))}
cand = {t: [] for t in KINDS}          # (file, query key, [answer keys])
for f in FILES:
    d = pd.read_parquet(f, columns=['key', 'created', 'issuelinks'])
    d['created'] = pd.to_datetime(d['created'], errors='coerce', utc=True)
    d = d[d['created'] >= pd.Timestamp('2000-01-01', tz='UTC')]
    when = dict(zip(d['key'], d['created']))
    for k, c, links in zip(d['key'], d['created'], d['issuelinks']):
        if c < q80: continue
        for t, match in KINDS.items():
            ans = [l['key'] for l in json.loads(links)
                   if match((l.get('type') or '').lower(), (l.get('label') or '').lower()) and l.get('key') in when and when[l['key']] < c]
            if ans: cand[t].append((f.name, k, ans))
    del d, when; gc.collect()
sel = {}
for t in KINDS:
    idx = rng.choice(len(cand[t]), size=min(N_TEST, len(cand[t])), replace=False)
    sel[t] = [cand[t][i] for i in sorted(idx)]
    pd.DataFrame({'key': [k for _, k, _ in sel[t]], 'answers': [json.dumps(a) for _, _, a in sel[t]]}).to_csv(OUTP/f'testset_jira_{t}.csv', index=False)
    print(t, len(cand[t]), 'queries available,', len(sel[t]), 'sampled')

## 4. I1 lexical search
TF-IDF cosine similarity over earlier issues of the same repository, one repository at a time. Hashed features keep memory low. Recall@1/5/10 and MRR@10.

In [ ]:
hv = HashingVectorizer(n_features=2**19, ngram_range=(1, 2), alternate_sign=False, norm=None, dtype=np.float32)
hits = {t: {1: 0, 5: 0, 10: 0} for t in KINDS}; rr = {t: [] for t in KINDS}; out = {t: {} for t in KINDS}; t0 = time.time()
for f in FILES:
    qs = {t: [(k, a) for fn, k, a in sel[t] if fn == f.name] for t in KINDS}
    if not any(qs.values()): continue
    d = pd.read_parquet(f, columns=['key', 'created', 'summary', 'description'])
    d['created'] = pd.to_datetime(d['created'], errors='coerce', utc=True)
    d = d[d['created'] >= pd.Timestamp('2000-01-01', tz='UTC')].reset_index(drop=True)
    text = (d['summary'].fillna('') + '. ' + d['description'].fillna('').str.slice(0, 1200)).values
    keys, cr = d['key'].values, d['created'].values.astype('datetime64[ns]').astype(np.int64)
    del d; gc.collect()
    H = hv.transform(text); del text; gc.collect()
    X = normalize(TfidfTransformer(sublinear_tf=True).fit(H[cr < q80.value]).transform(H)).tocsr(); del H; gc.collect()
    pos = {k: i for i, k in enumerate(keys)}
    for t in KINDS:
        q = qs[t]
        for b in range(0, len(q), 32):
            batch = q[b:b+32]; S = (X @ X[[pos[k] for k, _ in batch]].T).toarray()
            for j, (k, ans) in enumerate(batch):
                s = S[:, j]; s[cr >= cr[pos[k]]] = -1                      # only issues created earlier
                top = np.argpartition(-s, TOP_K)[:TOP_K]; top = top[np.argsort(-s[top])]
                ranks = [r+1 for r, i in enumerate(top) if keys[i] in set(ans)]
                for kk in hits[t]: hits[t][kk] += bool(ranks and ranks[0] <= kk)
                rr[t].append(1/ranks[0] if ranks else 0); out[t][k] = json.dumps([keys[i] for i in top])
    print(f.name, 'searched', f'{time.time()-t0:.0f}s')
    del X, keys, cr, pos; gc.collect()
rows = []
for t in KINDS:
    keys_t = [k for _, k, _ in sel[t]]
    pd.DataFrame({'key': keys_t, 'i1_top10': [out[t][k] for k in keys_t]}).to_csv(OUTP/f'i1_predictions_{t}.csv', index=False)
    n = len(keys_t)
    rows.append({'Decision': t, 'Model': 'I1 TF-IDF cosine (lexical search)', 'Queries available': len(cand[t]), 'Test rows': n,
                 'Recall@1 %': round(100*hits[t][1]/n, 1), 'Recall@5 %': round(100*hits[t][5]/n, 1), 'Recall@10 %': round(100*hits[t][10]/n, 1),
                 'Recall@10 95% CI': wilson(hits[t][10], n), 'MRR@10': round(float(np.mean(rr[t])), 3), 'Search s': round(time.time()-t0, 1)})
ret = pd.DataFrame(rows); ret.to_csv(RESULTS/'rq1_jira_i1_retrieval.csv', index=False); ret

## 5. Save ticket text and the step 04 summary

In [ ]:
need = set()
for t in KINDS:
    for _, k, a in sel[t]: need |= {k, *a}
cols = ['repository','key','project','issuetype','priority','resolution','created','summary','description','components']
old = pd.read_parquet(OUTP/'jira_test_tickets.parquet')
new = pd.concat([pd.read_parquet(f, columns=cols, filters=[('key', 'in', list(need - set(old['key'])))]) for f in FILES])
tick = pd.concat([old, new]).drop_duplicates('key'); tick.to_parquet(OUTP/'jira_test_tickets.parquet', index=False)
cls = pd.read_csv(RESULTS/'rq1_jira_i1_full.csv')
def mdt(d):
    try: return d.to_markdown(index=False)
    except Exception: return d.to_string()
lines = [f'# Step 04 summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', '', '## Classification (I1)', mdt(cls), '', '## Retrieval (I1)', mdt(ret)]
(RESULTS/'step04_summary.md').write_text('\n'.join(lines)); print(len(tick), 'tickets in the text file'); print('\n'.join(lines))